In [ ]:
!pip install dm-sonnet tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.4/268.4 kB 6.6 MB/s eta 0:00:00


In [ ]:
# Get enformer source code
!wget -q https://raw.githubusercontent.com/deepmind/deepmind-research/master/enformer/attention_module.py
!wget -q https://raw.githubusercontent.com/deepmind/deepmind-research/master/enformer/enformer.py

In [ ]:
import tensorflow as tf
import sonnet as snt
from tqdm import tqdm
import numpy as np
import os
import glob
import sys
from typing import Any, Callable, Dict, Optional, Text, Union, Iterable

import enformer

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Get references to original classes
OriginalSequential = enformer.Sequential
OriginalResidual = enformer.Residual

# Define a new, fixed Sequential class
class FixedSequential(snt.Module):
    """
    A fixed version of enformer.Sequential that correctly
    propagates the `is_training` argument.

    The original `accepts_is_training` check is unreliable
    due to Sonnet/TF wrappers. This version tries to pass
    `is_training` to all sub-modules using a try/except
    block, which is much more robust.
    """

    def __init__(self,
                 layers: Optional[Union[Callable[[], Iterable[snt.Module]],
                                        Iterable[Callable[..., Any]]]] = None,
                 name: Optional[Text] = None):
        super().__init__(name=name)
        if layers is None:
            self._layers = []
        else:
            if hasattr(layers, '__call__'):
                layers = layers()
            self._layers = [layer for layer in layers if layer is not None]

    def __call__(self, inputs: tf.Tensor, is_training: bool, **kwargs):
        outputs = inputs

        for _, mod in enumerate(self._layers):
            try:
                # Try to call the module WITH is_training
                outputs = mod(outputs, is_training=is_training, **kwargs)
            except TypeError as e:
                # If it fails, it's likely a function (like gelu)
                # or module (like Conv1D) that doesn't accept it.
                if 'is_training' in str(e) or 'unexpected keyword' in str(e):
                    # Call it again WITHOUT is_training
                    outputs = mod(outputs, **kwargs)
                else:
                    # It's a different, real error
                    raise e
        return outputs

# Patch the enformer module
enformer.Sequential = FixedSequential
print("Successfully patched enformer.Sequential")

Successfully patched enformer.Sequential


In [ ]:
np.random.seed(42)

In [ ]:
BATCH_SIZE = 8
SEQUENCE_LENGTH = 196608
OUTPUT_BINS = 896

# Paper-aligned settings
TARGET_LEARNING_RATE = 1e-4
ADAM_BETA1 = 0.9
ADAM_BETA2 = 0.999
ADAM_EPSILON = 1e-8
GRAD_CLIP_NORM = 0.2

# The paper used 5k warmup steps for 150k total train steps (~3%)
NUM_TRAIN_EPOCHS = 30
WARMUP_STEPS = None
PATIENCE_FOR_EARLY_STOPPING = 3

In [ ]:
def _load_npy_files(x_path_tensor):
    x_path = x_path_tensor.numpy().decode('utf-8')
    y_path = x_path.replace('X_', 'Y_')
    x = np.load(x_path).astype(np.float32)
    y = np.load(y_path).astype(np.float32)
    return x, y

@tf.function
def _load_data(x_path_tensor, is_training):
    x, y = tf.py_function(
        _load_npy_files,
        [x_path_tensor],
        [tf.float32, tf.float32]
    )

    if is_training:
        # Randomly reverse-complement
        if tf.random.uniform(shape=[]) > 0.5:
            x = tf.reverse(x, axis=[0, 1])  # Reverses seq and one-hot
            y = tf.reverse(y, axis=[0])      # Reverses the 896 bins

    x.set_shape([SEQUENCE_LENGTH, 4])
    y.set_shape([OUTPUT_BINS])
    return x, y

def create_dataset(directory_path, batch_size, is_training):
    x_files = glob.glob(os.path.join(directory_path, 'X_*.npy'))
    if not x_files:
        raise FileNotFoundError(f"No 'X_*.npy' files found in {directory_path}")

    dataset = tf.data.Dataset.from_tensor_slices(x_files)

    if is_training:
        dataset = dataset.shuffle(buffer_size=len(x_files)).repeat()

    dataset = dataset.map(
        lambda x_path: _load_data(x_path, is_training),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

Base Model

In [ ]:
checkpoint_gs_path = 'gs://dm-enformer/models/enformer/sonnet_weights/*'
checkpoint_path = '/tmp/enformer_checkpoint'

In [ ]:
!mkdir /tmp/enformer_checkpoint

In [ ]:
for file_path in tf.io.gfile.glob(checkpoint_gs_path):
    print(file_path)
    file_name = os.path.basename(file_path)
    tf.io.gfile.copy(file_path, f'{checkpoint_path}/{file_name}', overwrite=True)

gs://dm-enformer/models/enformer/sonnet_weights/checkpoint
gs://dm-enformer/models/enformer/sonnet_weights/enformer-fine-tuned-human-1.data-00000-of-00001
gs://dm-enformer/models/enformer/sonnet_weights/enformer-fine-tuned-human-1.index


In [ ]:
enformer_model = enformer.Enformer()

In [ ]:
checkpoint = tf.train.Checkpoint(module=enformer_model)

In [ ]:
latest = tf.train.latest_checkpoint(checkpoint_path)
status = checkpoint.restore(latest)

In [ ]:
def get_exponential_decay_encoding(num_bins, center_bin, scales):
    """
    Generates exponential decay features centered at the TSS.

    Args:
        num_bins: Total number of bins (896).
        center_bin: The index of the TSS (approx 448).
        scales: A list of 'A' values or length scales (e.g., [10, 50, 100 bins]).

    Returns:
        Tensor of shape (1, num_bins, len(scales))
    """
    # 1. Create absolute distance from center: |x - center|
    # Shape: (896,)
    positions = np.abs(np.arange(num_bins) - center_bin)

    encoding_list = []

    # 2. Generate a decay curve for each scale
    # Formula: exp( - distance / scale )
    # (Professor suggested A * dist, which is same as dist / (1/A))
    for scale in scales:
        # A small scale (e.g., 5 bins) decays very fast.
        # A large scale (e.g., 100 bins) decays slowly.
        decay_curve = np.exp(-positions / scale)
        encoding_list.append(decay_curve)

    # 3. Stack them into a feature matrix
    # Shape: (num_bins, num_scales)
    encoding_matrix = np.stack(encoding_list, axis=-1)

    # 4. Add batch dimension: (1, num_bins, num_scales)
    return tf.constant(encoding_matrix[np.newaxis, ...], dtype=tf.float32)

In [ ]:
class EnformerLinearHead(snt.Module):
    def __init__(self, base_model, output_bins, name='enformer_linear_head'):
        super().__init__(name=name)
        self._base = base_model
        self._head = snt.Linear(output_bins)

        # Define the "A" values (scales)
        # These represent "decay length" in bins (1 bin = 128bp)
        # 1 bin  (~128bp)   -> Very short range
        # 10 bins (~1.2kb)  -> Promoter region
        # 100 bins (~12kb)  -> Enhancer region
        # 500 bins (~64kb)  -> Long range
        decay_scales = [1.0, 5.0, 10.0, 50.0, 100.0, 200.0, 500.0]

        self.pos_embedding = get_exponential_decay_encoding(
            num_bins=896,
            center_bin=447.5,
            scales=decay_scales
        )

    def __call__(self, x, is_training):
        # 1. Get Enformer embeddings
        # Shape: (batch_size, 896, 5313)
        base_output = self._base(x, is_training=is_training)['human']

        # 2. Prepare Position Features
        # The static embedding has shape (1, 896, 7).
        # We need to copy it to match the current batch size.
        batch_size = tf.shape(base_output)[0]

        # tf.tile copies the tensor:
        # - [batch_size] copies along axis 0
        # - [1] keeps axis 1 (bins) same
        # - [1] keeps axis 2 (features) same
        # Result Shape: (batch_size, 896, 7)
        pos_feat = tf.tile(self.pos_embedding, [batch_size, 1, 1])

        # 3. Concatenate ("Glue") them together
        # We join them along axis -1 (the feature axis).
        # Input A: (batch, 896, 5313)
        # Input B: (batch, 896, 7)
        # Result:  (batch, 896, 5320)
        augmented_input = tf.concat([base_output, pos_feat], axis=-1)

        # 4. Pass to Linear Head
        # The linear layer now sees [Bio_1, ..., Bio_5313, Decay_1, ..., Decay_7]
        head_output = self._head(augmented_input)

        return tf.squeeze(head_output, axis=-1)

In [ ]:
@tf.function
def train_step(model, optimizer, loss_fn, batch_x, batch_y):
    with tf.GradientTape() as tape:
        outputs = model(batch_x, True)
        loss = loss_fn(batch_y, outputs)

    trainable_vars = []

    trainable_vars.extend(model._head.trainable_variables)

    for var in model._base.trainable_variables:
        if 'transformer_block_10' in var.name:
            trainable_vars.append(var)

    gradients = tape.gradient(loss, trainable_vars)

    optimizer.apply_gradients(zip(gradients, trainable_vars))

    return loss

@tf.function
def valid_step(model, loss_fn, batch_x, batch_y):
    outputs = model(batch_x, False)
    loss = loss_fn(batch_y, outputs)
    return loss

In [ ]:
def balanced_mse_loss(y_true, y_pred):
    """
    Calculates MSE separately for peaks (non-zeros) and background (zeros),
    then averages them 50/50.
    """
    # Flatten the data to make masking easier
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])

    # Create Masks
    # Peaks: Anywhere the true value is NOT zero
    peak_mask = tf.not_equal(y_true_f, 0)
    # Background: Anywhere the true value IS zero
    bg_mask = tf.equal(y_true_f, 0)

    # Separate the data
    peaks_true = tf.boolean_mask(y_true_f, peak_mask)
    peaks_pred = tf.boolean_mask(y_pred_f, peak_mask)

    bg_true = tf.boolean_mask(y_true_f, bg_mask)
    bg_pred = tf.boolean_mask(y_pred_f, bg_mask)

    # Calculate MSE for Background
    # (We assume there is always *some* background in a batch)
    bg_loss = tf.reduce_mean(tf.square(bg_true - bg_pred))

    # Calculate MSE for Peaks (Safe Handling)
    # Check if we actually found any peaks in this batch
    num_peaks = tf.shape(peaks_true)[0]

    def compute_peak_loss():
        return tf.reduce_mean(tf.square(peaks_true - peaks_pred))

    def zero_loss():
        return tf.constant(0.0, dtype=tf.float32)

    # If peaks exist, calc MSE. If not, return 0.0
    peak_loss = tf.cond(num_peaks > 0, compute_peak_loss, zero_loss)

    # Combine 50/50
    # If the batch had peaks, we weight 50/50.
    # If the batch had NO peaks, we just use the background loss (100%).

    total_loss = tf.cond(
        num_peaks > 0,
        lambda: 0.5 * peak_loss + 0.5 * bg_loss,
        lambda: bg_loss
    )

    return total_loss

In [ ]:
# --- Un-tar the data ---
gdrive_tar_path = '/content/drive/MyDrive/Junior/CRE-Prediction/output_regression.tar.gz'

# Path from data prep script
local_data_path = 'output_samples_regression/'

# Only un-tar if the folder doesn't already exist
if not os.path.exists(local_data_path):
    print(f"Data folder not found. Unpacking {gdrive_tar_path}...")
    # This runs the 'tar' command on the Colab VM
    # -x = extract, -z = un-gzip, -f = from file
    os.system(f'tar -xzf "{gdrive_tar_path}"')
    print("Data successfully unpacked to local disk.")
else:
    print("Data folder already exists locally.")

DATA_PATH = local_data_path
TRAIN_PATH = os.path.join(DATA_PATH, 'train')
VALID_PATH = os.path.join(DATA_PATH, 'validation')
TEST_PATH = os.path.join(DATA_PATH, 'test')

# --- Load Data & Get Sizes ---
print(f"Checking data paths in {DATA_PATH}...")
num_train = len(glob.glob(os.path.join(TRAIN_PATH, 'X_*.npy')))
num_valid = len(glob.glob(os.path.join(VALID_PATH, 'X_*.npy')))
num_test = len(glob.glob(os.path.join(TEST_PATH, 'X_*.npy')))

if num_train == 0:
    print(f"Error: No training files found in {TRAIN_PATH}", file=sys.stderr)
    sys.exit(1)

print(f"Found {num_train} train, {num_valid} validation, {num_test} test files.")

# --- Calculate training steps ---
STEPS_PER_EPOCH = num_train // BATCH_SIZE
WARMUP_STEPS = STEPS_PER_EPOCH # Use 1 epoch for warmup
VALIDATION_STEPS = (num_valid + BATCH_SIZE - 1) // BATCH_SIZE
print(f"Using BATCH_SIZE = {BATCH_SIZE}")
print(f"Train: {num_train} samples, {STEPS_PER_EPOCH} steps per epoch.")
print(f"Valid: {num_valid} samples, {VALIDATION_STEPS} steps per validation.")

# --- Create Datasets ---
print("Creating tf.data.Datasets...")
train_dataset = create_dataset(TRAIN_PATH, BATCH_SIZE, True)
valid_dataset = create_dataset(VALID_PATH, BATCH_SIZE, False)

# --- Create model ---
model = EnformerLinearHead(enformer_model, 1)

# --- Setup Optimizer, LR Scheduler, and Loss ---
lr_schedule = tf.keras.optimizers.schedules.PolynomialDecay(
    0.0,
    WARMUP_STEPS,
    TARGET_LEARNING_RATE,
    1.0
)

optimizer = tf.keras.optimizers.Adam(
    learning_rate=lr_schedule,
    beta_1=ADAM_BETA1,
    beta_2=ADAM_BETA2,
    epsilon=ADAM_EPSILON,
    clipnorm=GRAD_CLIP_NORM
)

loss_fn = balanced_mse_loss

# --- Run the Training Loop ---
print(f"--- Starting training for {NUM_TRAIN_EPOCHS} epochs ---")

# Early stopping variables
best_valid_loss = float('inf')
patience_counter = 0
best_model_path = 'best_model_checkpoint'

checkpoint = tf.train.Checkpoint(module=model)
manager = tf.train.CheckpointManager(checkpoint, best_model_path, max_to_keep=1)

train_iter = iter(train_dataset)

for epoch in range(NUM_TRAIN_EPOCHS):
    print(f"\n--- Epoch {epoch + 1}/{NUM_TRAIN_EPOCHS} ---")

    # Training
    train_loss_total = 0
    for i in tqdm(range(STEPS_PER_EPOCH), desc="Training"):
        x_train, y_train = next(train_iter)
        train_loss = train_step(model, optimizer, loss_fn, x_train, y_train)
        train_loss_total += train_loss.numpy()

    avg_train_loss = train_loss_total / STEPS_PER_EPOCH

    # Validation
    valid_loss_total = 0

    # Re-initialize validation iterator
    valid_iter = iter(valid_dataset)
    for i in tqdm(range(VALIDATION_STEPS), desc="Validating"):
        x_valid, y_valid = next(valid_iter)
        valid_loss_total += valid_step(model, loss_fn, x_valid, y_valid).numpy()

    avg_valid_loss = valid_loss_total / VALIDATION_STEPS

    print(f"Epoch {epoch + 1} | Train Loss: {avg_train_loss:.6f} | Valid Loss: {avg_valid_loss:.6f}")

    # Early stopping logic
    if avg_valid_loss < best_valid_loss:
        best_valid_loss = avg_valid_loss
        patience_counter = 0
        print(f"New best model found! Saving to {best_model_path}")
        manager.save()
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{PATIENCE_FOR_EARLY_STOPPING}")

    if patience_counter >= PATIENCE_FOR_EARLY_STOPPING:
        print("Early stopping triggered. Halting training.")
        break

print("\n--- Training complete ---")

# --- Final Evaluation ---
# Define evaluation helpers
@tf.function
def correlation_coefficient(y_true, y_pred):
    x = y_true
    y = y_pred
    mx = tf.reduce_mean(x)
    my = tf.reduce_mean(y)
    xm, ym = x - mx, y - my
    r_num = tf.reduce_sum(xm * ym)
    r_den = tf.sqrt(tf.reduce_sum(tf.square(xm)) * tf.reduce_sum(tf.square(ym)))
    if r_den == 0:
        return 0.0
    return r_num / r_den

@tf.function
def predict_batch(model, x):
    return model(x, False)

# Restore model
print(f"\nLoading best model from {best_model_path} for final evaluation...")
eval_model = EnformerLinearHead(enformer_model, 1)
eval_checkpoint = tf.train.Checkpoint(module=eval_model)
status = eval_checkpoint.restore(tf.train.latest_checkpoint(best_model_path))
status.assert_existing_objects_matched()

print("Running evaluation on the TEST set...")

# Re-create the test dataset
test_dataset_eval = create_dataset(TEST_PATH, BATCH_SIZE, False)
num_batches = num_test // BATCH_SIZE

# Metrics storage
all_true_masked = []
all_pred_masked = []
all_true_full = []
all_pred_full = []
test_loss_total = 0.0

# Evaluation Loop
for x_test, y_test in tqdm(test_dataset_eval.take(num_batches),
                            total=num_batches,
                            desc="Evaluating Test Set"):

    # Get predictions
    y_pred = predict_batch(eval_model, x_test)

    # Calculate & Accumulate Loss (MSE)
    batch_loss = loss_fn(y_test, y_pred)
    test_loss_total += batch_loss.numpy()

    # Flatten Data
    y_true_flat = tf.reshape(y_test, [-1])
    y_pred_flat = tf.reshape(y_pred, [-1])

    # Store full Data (For Overall R)
    all_true_full.append(y_true_flat)
    all_pred_full.append(y_pred_flat)

    # Store masked Data (For Peaks R)
    mask = tf.where(y_true_flat != 0)
    y_true_masked = tf.gather(y_true_flat, mask)
    y_pred_masked = tf.gather(y_pred_flat, mask)

    all_true_masked.append(y_true_masked)
    all_pred_masked.append(y_pred_masked)

# --- Compute Final Metrics ---
# Calculate Average MSE Loss
avg_test_loss = test_loss_total / num_batches

# Calculate Global Pearson R (Peaks Only)
full_y_true_masked = tf.concat(all_true_masked, axis=0)
full_y_pred_masked = tf.concat(all_pred_masked, axis=0)
pearson_peaks = correlation_coefficient(full_y_true_masked, full_y_pred_masked)

# Calculate Global Pearson R (Overall / All Data)
full_y_true_all = tf.concat(all_true_full, axis=0)
full_y_pred_all = tf.concat(all_pred_full, axis=0)
pearson_overall = correlation_coefficient(full_y_true_all, full_y_pred_all)

print("\n" + "="*40)
print("FINAL TEST RESULTS")
print("="*40)
print(f"Test MSE Loss:      {avg_test_loss:.6f}")
print("-" * 40)
print(f"Test Pearson R (Peaks Only):  {pearson_peaks.numpy():.4f}")
print(f"Test Pearson R (Overall):     {pearson_overall.numpy():.4f}")
print("="*40)

Data folder not found. Unpacking /content/drive/MyDrive/Junior/MLFG/output_regression.tar.gz...
Data successfully unpacked to local disk.
Checking data paths in output_samples_regression/...
Found 6329 train, 885 validation, 787 test files.
Using BATCH_SIZE = 8
Train: 6329 samples, 791 steps per epoch.
Valid: 885 samples, 111 steps per validation.
Creating tf.data.Datasets...
--- Starting training for 30 epochs ---

--- Epoch 1/30 ---


Validating: 100%|██████████| 111/111 [05:11<00:00,  2.81s/it]


Epoch 1 | Train Loss: 5.221068 | Valid Loss: 0.355531
New best model found! Saving to best_model_checkpoint

--- Epoch 2/30 ---


Validating: 100%|██████████| 111/111 [03:40<00:00,  1.99s/it]


Epoch 2 | Train Loss: 0.117106 | Valid Loss: 0.060089
New best model found! Saving to best_model_checkpoint

--- Epoch 3/30 ---


...

Epoch 22 | Train Loss: 0.031116 | Valid Loss: 0.034159
No improvement. Patience: 2/3

--- Epoch 23/30 ---


Validating: 100%|██████████| 111/111 [03:40<00:00,  1.99s/it]


Epoch 23 | Train Loss: 0.029879 | Valid Loss: 0.034654
No improvement. Patience: 3/3
Early stopping triggered. Halting training.

--- Training complete ---

Loading best model from best_model_checkpoint for final evaluation...
Running evaluation on the TEST set...


Evaluating Test Set: 100%|██████████| 98/98 [03:27<00:00,  2.12s/it]


FINAL TEST RESULTS
Test MSE Loss:      0.034429
----------------------------------------
Test Pearson R (Peaks Only):  0.6168
Test Pearson R (Overall):     0.2086
